# 05 — Promotion Uplift Modeling

## Objective

This notebook estimates the incremental effect of retail promotions on customer demand.

The analysis will:

- Define promotion participation as the treatment variable.
- Estimate baseline demand without promotion.
- Measure the additional units sold because of promotion.
- Compare promoted and non-promoted observations.
- Estimate average and heterogeneous treatment effects.
- Identify products, stores, and customer segments with the highest expected uplift.
- Evaluate promotion effectiveness using uplift-based metrics.
- Save promotion-uplift results for forecasting and price optimization.

## Treatment and Outcome Definition

- **Treatment:** Promotion is active (`promotion = 1`)
- **Control:** Promotion is not active (`promotion = 0`)
- **Primary outcome:** Units sold
- **Secondary outcomes:** Revenue and estimated contribution margin

This notebook distinguishes true incremental promotion impact from ordinary differences between promoted and non-promoted sales.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Locate the project root
current_path = Path.cwd()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "data").exists():
    PROJECT_ROOT = current_path.parent
else:
    raise FileNotFoundError(
        "Project root could not be located. Confirm that the notebook is inside the project folder."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

print("\nAvailable CSV files:")
for file_path in sorted(PROJECT_ROOT.glob("data/**/*.csv")):
    print(f"- {file_path.relative_to(PROJECT_ROOT)}")

Project root: c:\Users\marja\OneDrive\Desktop\GitHub\retail-pricing-promotion-optimization
Raw data directory: c:\Users\marja\OneDrive\Desktop\GitHub\retail-pricing-promotion-optimization\data\raw
Processed data directory: c:\Users\marja\OneDrive\Desktop\GitHub\retail-pricing-promotion-optimization\data\processed

Available CSV files:
- data\processed\iv_2sls_model_comparison.csv
- data\processed\iv_first_stage_diagnostics.csv
- data\processed\iv_specification_diagnostics.csv
- data\processed\product_elasticity_estimates.csv
- data\raw\m5\calendar.csv
- data\raw\m5\sales_train_validation.csv
- data\raw\m5\sell_prices.csv
- data\raw\synthetic\market_notes.csv
- data\raw\synthetic\sample_retail_pricing.csv


In [2]:
# Inspect CSV files and identify datasets suitable for promotion-uplift modeling

csv_files = sorted(PROJECT_ROOT.glob("data/**/*.csv"))

promotion_keywords = {
    "promotion",
    "promo",
    "is_promotion",
    "promotion_flag",
    "discount",
    "discount_rate",
}

outcome_keywords = {
    "units_sold",
    "sales",
    "quantity",
    "demand",
    "revenue",
}

candidate_datasets = []

for file_path in csv_files:
    try:
        sample = pd.read_csv(file_path, nrows=5)
        normalized_columns = {
            column.strip().lower().replace(" ", "_")
            for column in sample.columns
        }

        promotion_columns = normalized_columns.intersection(promotion_keywords)
        outcome_columns = normalized_columns.intersection(outcome_keywords)

        if promotion_columns and outcome_columns:
            candidate_datasets.append(
                {
                    "file": str(file_path.relative_to(PROJECT_ROOT)),
                    "promotion_columns": sorted(promotion_columns),
                    "outcome_columns": sorted(outcome_columns),
                    "number_of_columns": len(sample.columns),
                }
            )

    except Exception as error:
        print(f"Could not inspect {file_path.name}: {error}")

candidate_df = pd.DataFrame(candidate_datasets)

if candidate_df.empty:
    print("No clear promotion dataset was identified.")
    print("Review the available column names manually.")
else:
    display(candidate_df)

,file,promotion_columns,outcome_columns,number_of_columns
0,data\raw\synthetic\sample_retail_pricing.csv,[promotion_flag],"[revenue, units_sold]",29


In [3]:
# Load the synthetic retail dataset used for promotion-uplift modeling

INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

candidate_paths = [
    RAW_DATA_DIR / "synthetic" / "sample_retail_pricing.csv",
    RAW_DATA_DIR / "sample_retail_pricing.csv",
    INTERIM_DATA_DIR / "retail_demand_clean.csv",
    INTERIM_DATA_DIR / "retail_data_initial.csv",
]

data_path = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if data_path is None:
    raise FileNotFoundError(
        "The retail dataset could not be found. "
        "Expected sample_retail_pricing.csv in data/raw/synthetic."
    )

df = pd.read_csv(data_path)

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(f"Loaded dataset: {data_path.relative_to(PROJECT_ROOT)}")
print(f"Original shape: {df.shape}")

# Validate required columns
required_columns = {
    "date",
    "store_id",
    "product_id",
    "promotion",
    "quantity",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

# Convert the date column
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Standardize the promotion treatment variable
if df["promotion"].dtype == "object":
    promotion_mapping = {
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "active": 1,
        "inactive": 0,
        "promotion": 1,
        "no_promotion": 0,
    }

    df["promotion"] = (
        df["promotion"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(promotion_mapping)
    )

df["promotion"] = pd.to_numeric(
    df["promotion"],
    errors="coerce",
)

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce",
)

# Keep valid observations
df = (
    df.dropna(
        subset=[
            "date",
            "store_id",
            "product_id",
            "promotion",
            "quantity",
        ]
    )
    .copy()
)

df["promotion"] = df["promotion"].astype(int)

# Promotion must be binary
invalid_treatment_values = set(df["promotion"].unique()).difference({0, 1})

if invalid_treatment_values:
    raise ValueError(
        "The promotion column must contain only 0 and 1. "
        f"Invalid values: {sorted(invalid_treatment_values)}"
    )

df = df.sort_values("date").reset_index(drop=True)

print(f"Validated shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Stores: {df['store_id'].nunique():,}")
print(f"Products: {df['product_id'].nunique():,}")
print(f"Promotion rate: {df['promotion'].mean():.2%}")

print("\nTreatment distribution:")
display(
    df["promotion"]
    .value_counts()
    .rename_axis("promotion")
    .reset_index(name="observations")
)

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Loaded dataset: data\raw\synthetic\sample_retail_pricing.csv
Original shape: (18300, 29)


ValueError: Required columns are missing: ['promotion', 'quantity']

In [4]:
print(df.columns.tolist())

['date', 'store_id', 'region', 'product_id', 'category', 'year', 'month', 'day_of_week', 'weekend_flag', 'holiday_flag', 'regular_price', 'selling_price', 'competitor_price', 'unit_cost', 'discount_pct', 'promotion_flag', 'promotion_type', 'promotion_propensity', 'advertising_spend', 'supplier_cost_index', 'shipping_cost_index', 'weather_index', 'customer_traffic', 'inventory_level', 'stockout_flag', 'units_sold', 'revenue', 'gross_profit', 'true_price_elasticity']


In [5]:
# Automatically identify important columns using common alternative names

column_aliases = {
    "date": [
        "date",
        "transaction_date",
        "sales_date",
        "week_date",
        "day",
    ],
    "store_id": [
        "store_id",
        "store",
        "store_code",
        "location_id",
    ],
    "product_id": [
        "product_id",
        "item_id",
        "sku_id",
        "sku",
        "product",
        "item",
    ],
    "promotion": [
        "promotion",
        "promo",
        "promo_flag",
        "promotion_flag",
        "is_promotion",
        "is_promo",
        "on_promotion",
        "promo_active",
        "discount_flag",
    ],
    "quantity": [
        "quantity",
        "units_sold",
        "unit_sales",
        "sales_quantity",
        "demand",
        "qty",
        "units",
    ],
}

detected_columns = {}

for standard_name, possible_names in column_aliases.items():
    match = next(
        (column for column in possible_names if column in df.columns),
        None,
    )

    if match is not None:
        detected_columns[standard_name] = match

print("Detected column mapping:")
for standard_name, original_name in detected_columns.items():
    print(f"- {standard_name}: {original_name}")

missing_standard_columns = set(column_aliases).difference(detected_columns)

if missing_standard_columns:
    print(
        "\nColumns that could not be identified:",
        sorted(missing_standard_columns),
    )
    print("\nAvailable dataset columns:")
    print(df.columns.tolist())
else:
    rename_mapping = {
        original_name: standard_name
        for standard_name, original_name in detected_columns.items()
        if original_name != standard_name
    }

    df = df.rename(columns=rename_mapping)

    print("\nAll required columns were successfully identified.")
    print(df[["date", "store_id", "product_id", "promotion", "quantity"]].head())

Detected column mapping:
- date: date
- store_id: store_id
- product_id: product_id
- promotion: promotion_flag
- quantity: units_sold

All required columns were successfully identified.
         date store_id product_id  promotion  quantity
0  2024-01-01     S001       P001          0        47
1  2024-01-01     S001       P002          0        50
2  2024-01-01     S001       P003          0        26
3  2024-01-01     S001       P004          0        91
4  2024-01-01     S001       P005          0        54
